**Recovery note:** This notebook was reconstructed from an HTML export. The code, narrative, and visible outputs were preserved where possible.


<h1 id="ETA-Discrepancy-Detection">ETA Discrepancy Detection<a class="anchor-link" href="#ETA-Discrepancy-Detection">¶</a></h1><p>This notebook demonstrates how the trained ETA prediction model can be used to identify discrepancies between the reported ETA transmitted through AIS and the ETA estimated by the machine learning model.</p>
<p>For each vessel observation, the predicted remaining travel time is converted into a predicted ETA and compared with the reported ETA. Large differences are flagged as potential ETA discrepancies.</p>


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor


<h3 id="Retrain-the-Selected-Model">Retrain the Selected Model<a class="anchor-link" href="#Retrain-the-Selected-Model">¶</a></h3><p>Notebook 3 showed that LightGBM consistently outperformed Linear Regression across all evaluation strategies.
For the sake of this exercise, LightGBM is retrained using the same configuration and used to generate ETA predictions for discrepancy detection.</p>


In [10]:
# Load the processed dataset
data = pd.read_csv(
    "../data/processed/model_data_with_metadata.csv",
    parse_dates=["recorded_at", "eta_datetime"]
)

# Prepare features and target
metadata_columns = ["mmsi", "recorded_at", "eta_datetime"]

X = data.drop(columns=metadata_columns + ["remaining_hours"])
y = data["remaining_hours"]

# Random train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train the LightGBM model
model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

# Predict the remaining travel time
predicted_remaining_hours = model.predict(X_test)


<h2 id="Calculate-ETA-Discrepancies">Calculate ETA Discrepancies<a class="anchor-link" href="#Calculate-ETA-Discrepancies">¶</a></h2><p>The predicted remaining travel time is converted into a predicted arrival timestamp. This predicted ETA is then compared with the ETA reported through AIS.</p>


In [11]:
# Recover metadata for the test data
results = data.loc[X_test.index, [
    "mmsi",
    "recorded_at",
    "eta_datetime"
]].copy()

# Add the model predictions
results["predicted_remaining_hours"] = predicted_remaining_hours
#display(results.head())

# remove negative predicted travel times
results["predicted_remaining_hours"] = results[
    "predicted_remaining_hours"
].clip(lower=0)

duration = pd.to_timedelta(
    results["predicted_remaining_hours"],
    unit="h"
)

display(duration.head())


# Convert predicted remaining hours into a predicted ETA
#3 April 10:42 + 327.8 hours = predicted ETA
results["predicted_eta"] = (
    results["recorded_at"]
    + pd.to_timedelta(
        results["predicted_remaining_hours"],
        unit="h"
    )
)   #retunrns 13 days 15 hours 48 minutes

# Calculate the absolute difference between predicted and reported ETA
results["difference_hours"] = (
    results["predicted_eta"] - results["eta_datetime"]
).dt.total_seconds().abs() / 3600 #conver seconds into hrs because Timedelta (a duration) give it to us in seconds

# Flag large discrepancies
discrepancy_threshold = 24   #################################################

results["discrepancy_flag"] = (
    results["difference_hours"] > discrepancy_threshold
)

results = results.sort_values(
    "difference_hours",
    ascending=False
)

results.head(10)


303332   1 days 20:43:04.718588757
189288   9 days 12:56:13.737824066
166635   3 days 11:55:54.761117466
263668   4 days 04:48:33.848472787
64924    6 days 03:00:29.108743653
Name: predicted_remaining_hours, dtype: timedelta64[ns]


,mmsi,recorded_at,eta_datetime,predicted_remaining_hours,predicted_eta,difference_hours,discrepancy_flag
271528,636024824,2026-04-12 12:09:26,2026-05-09 12:00:00,227.797755,2026-04-21 23:57:17.916959992,420.045023,True
244850,636024824,2026-04-12 02:41:47,2026-05-09 12:00:00,247.501999,2026-04-22 10:11:54.197825643,409.801612,True
244978,636024824,2026-04-12 02:44:50,2026-05-09 12:00:00,247.501999,2026-04-22 10:14:57.197825643,409.750778,True
245128,636024824,2026-04-12 02:48:55,2026-05-09 12:00:00,247.501999,2026-04-22 10:19:02.197825643,409.682723,True
245312,636024824,2026-04-12 02:54:00,2026-05-09 12:00:00,247.501999,2026-04-22 10:24:07.197825643,409.598001,True
25405,636024336,2026-04-04 23:30:29,2026-05-04 18:00:00,311.291348,2026-04-17 22:47:57.853570976,403.200596,True
299964,352506000,2026-04-15 08:56:23,2026-05-12 03:00:00,262.361549,2026-04-26 07:18:04.574923852,379.698729,True
300913,352506000,2026-04-15 10:40:56,2026-05-12 03:00:00,261.032634,2026-04-26 07:42:53.483076749,379.285144,True
300942,352506000,2026-04-15 10:43:59,2026-05-12 03:00:00,261.032634,2026-04-26 07:45:56.483076749,379.234310,True
26676,241287000,2026-04-05 01:00:37,2026-04-30 06:00:00,233.709513,2026-04-14 18:43:11.246832036,371.280209,True


<h2 id="Note:">Note:<a class="anchor-link" href="#Note:">¶</a></h2><ul>
<li>Discrepancies are calculated per observation. In production, flags wpuld be aggregated by vessel or create one alert per vessel over a defined time window.</li>
<li>24-hour was chosen for demonstration. In practice, the threshold could be adjusted depending on the use case</li>
</ul>


In [12]:
# Create a cleaner version of the results
results_display = results.copy()

results_display["predicted_remaining_hours"] = (
    results_display["predicted_remaining_hours"].round(1)
)

results_display["predicted_eta"] = (
    results_display["predicted_eta"].dt.round("min")
)

results_display["difference_hours"] = (
    results_display["difference_hours"].round(1)
)

# Summary
summary = pd.DataFrame({
    "Metric": [
        "Test observations",
        "Flagged discrepancies",
        "Flagged percentage"
    ],
    "Value": [
        f"{len(results_display):,}",
        f"{results_display['discrepancy_flag'].sum():,}",
        f"{results_display['discrepancy_flag'].mean() * 100:.1f}%"
    ]
})

display(summary)

# Show the 10 largest flagged discrepancies
results_display.loc[
    results_display["discrepancy_flag"]
].head(10)


,Metric,Value
0,Test observations,"62,412"
1,Flagged discrepancies,"23,515"
2,Flagged percentage,37.7%


,mmsi,recorded_at,eta_datetime,predicted_remaining_hours,predicted_eta,difference_hours,discrepancy_flag
271528,636024824,2026-04-12 12:09:26,2026-05-09 12:00:00,227.8,2026-04-21 23:57:00,420.0,True
244850,636024824,2026-04-12 02:41:47,2026-05-09 12:00:00,247.5,2026-04-22 10:12:00,409.8,True
244978,636024824,2026-04-12 02:44:50,2026-05-09 12:00:00,247.5,2026-04-22 10:15:00,409.8,True
245128,636024824,2026-04-12 02:48:55,2026-05-09 12:00:00,247.5,2026-04-22 10:19:00,409.7,True
245312,636024824,2026-04-12 02:54:00,2026-05-09 12:00:00,247.5,2026-04-22 10:24:00,409.6,True
25405,636024336,2026-04-04 23:30:29,2026-05-04 18:00:00,311.3,2026-04-17 22:48:00,403.2,True
299964,352506000,2026-04-15 08:56:23,2026-05-12 03:00:00,262.4,2026-04-26 07:18:00,379.7,True
300913,352506000,2026-04-15 10:40:56,2026-05-12 03:00:00,261.0,2026-04-26 07:43:00,379.3,True
300942,352506000,2026-04-15 10:43:59,2026-05-12 03:00:00,261.0,2026-04-26 07:46:00,379.2,True
26676,241287000,2026-04-05 01:00:37,2026-04-30 06:00:00,233.7,2026-04-14 18:43:00,371.3,True


<h3 id="Findings">Findings<a class="anchor-link" href="#Findings">¶</a></h3><ul>
<li>37.7% of the test observations were flagged because the difference between the model-predicted ETA and the AIS-reported ETA exceeded 24 hours.</li>
</ul>


<h2 id="Future-Improvements">Future Improvements<a class="anchor-link" href="#Future-Improvements">¶</a></h2><h3 id="Model-and-Data-Improvements">Model and Data Improvements<a class="anchor-link" href="#Model-and-Data-Improvements">¶</a></h3><ul>
<li><p>Create more movement variables, such as speed change:</p>
<p><code>Current SOG − Previous SOG</code></p>
</li>
<li><p>Tune the LightGBM model to improve prediction accuracy.</p>
</li>
<li><p>Improve the handling of missing or unrealistic values. For example, missing speed values could be filled using nearby observations from the same vessel.</p>
</li>
<li><p>Use a sequence-based model, such as an LSTM or Transformer, that looks at the vessel's recent movement instead of treating each AIS observation separately.</p>
</li>
</ul>
